# Notebook 01: Análisis Exploratorio de Datos (EDA) Completo
Este notebook realiza el análisis exploratorio exhaustivo del dataset **IBM Telco Customer Churn** para identificar patrones de comportamiento, relaciones entre variables y posibles factores que influyen en la fuga de clientes (*Churn*).

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar visualizaciones
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Ruta al dataset original
data_path = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(data_path)

print(f"Dimensiones originales: {df.shape[0]} filas, {df.shape[1]} columnas")
df.head()

## 1. Detección y Limpieza de Tipos de Datos y Valores Faltantes
La variable `TotalCharges` se lee comúnmente como texto porque tiene espacios vacíos (`" "`) para registros de clientes nuevos (`tenure == 0`). Procedemos a realizar la conversión a numérica y a inspeccionar los nulos reales.

In [ ]:
# Conversión de TotalCharges a numérico (coerción de errores a NaN)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Revisión de tipos de datos
print("--- Tipos de Datos ---")
print(df.dtypes)

# Detección de nulos reales
print("\n--- Conteo de Valores Nulos ---")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Ver registros nulos de TotalCharges
nulos_total_charges = df[df["TotalCharges"].isnull()]
print(f"Cantidad de nulos en TotalCharges: {len(nulos_total_charges)}")
nulos_total_charges[["tenure", "MonthlyCharges", "TotalCharges", "Churn"]].head()

Como se observa, todos los valores nulos en `TotalCharges` corresponden a clientes con `tenure == 0` (nuevos clientes que aún no han completado su primer ciclo de facturación). Dado que son muy pocos registros (11 de 7043), procedemos a eliminarlos.

In [ ]:
df.dropna(subset=["TotalCharges"], inplace=True)
df.drop(columns=["customerID"], inplace=True, errors="ignore")
print(f"Dimensiones después de remover nulos: {df.shape[0]} filas")

## 2. Análisis del Desbalance de la Variable Objetivo (Churn)

In [ ]:
# Proporción de Churn
churn_counts = df["Churn"].value_counts()
churn_pct = df["Churn"].value_counts(normalize=True) * 100

print("Distribución de Churn:")
for index, count in churn_counts.items():
    print(f"{index}: {count} ({churn_pct[index]:.2f}%)")

# Gráfico de barras de Churn
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Churn", palette="coolwarm")
plt.title("Distribución de la Variable Objetivo: Churn")
plt.show()

**Nota de negocio:** Existe un claro desbalanceo (~3:1). Solo un 26.58% de los clientes han abandonado la compañía. Esto requiere ajustar los modelos mediante técnicas de remuestreo o parámetros de peso (`scale_pos_weight` en XGBoost) para evitar que los clasificadores se sesguen a favor de la clase mayoritaria.

## 3. Análisis de Variables Numéricas y Correlaciones

In [ ]:
# Heatmap de correlación lineal completa (incluyendo TotalCharges)
plt.figure(figsize=(8, 6))
sns.heatmap(
    df.select_dtypes(include="number").corr(), 
    annot=True, 
    cmap="coolwarm", 
    fmt=".3f", 
    vmin=-1, 
    vmax=1
)
plt.title("Matriz de Correlación de Variables Numéricas")
plt.show()

**Observaciones:** Hay una alta correlación lineal positiva (0.826) entre `TotalCharges` y `tenure`, lo cual es lógico ya que a mayor antigüedad el cliente acumula mayores cargos totales. También hay correlación positiva (0.651) entre `TotalCharges` y `MonthlyCharges`.

In [ ]:
# Distribuciones de las variables numéricas clave
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, col in enumerate(num_cols):
    sns.histplot(data=df, x=col, hue="Churn", kde=True, multiple="stack", ax=axes[i], palette="coolwarm")
    axes[i].set_title(f"Distribución de {col} según Churn")

plt.tight_layout()
plt.show()

## 4. Análisis de Relación de Variables Categóricas clave con Churn

In [ ]:
# Variables categóricas estratégicas
cat_cols = ["Contract", "InternetService", "PaymentMethod"]

fig, axes = plt.subplots(3, 1, figsize=(10, 16))
for i, col in enumerate(cat_cols):
    sns.countplot(data=df, x=col, hue="Churn", ax=axes[i], palette="coolwarm")
    axes[i].set_title(f"Tasa de Churn por {col}")
    axes[i].tick_params(axis='x', rotation=15)
    
plt.tight_layout()
plt.show()

**Hallazgos Clave:**
1. **Tipo de Contrato:** Los clientes con contrato mes a mes (`Month-to-month`) muestran una tasa de deserción dramáticamente mayor en comparación con los contratos a 1 o 2 años.
2. **Servicio de Internet:** La deserción es notablemente más alta para los usuarios con internet de Fibra Óptica (`Fiber optic`), sugiriendo posibles descontentos con la tarifa o el servicio.
3. **Método de Pago:** Aquellos que pagan con cheque electrónico (`Electronic check`) tienen una deserción mucho mayor en comparación con métodos automáticos (tarjeta de crédito o transferencia bancaria).